<a href="https://colab.research.google.com/github/AtziriMendoza/API-challenge2/blob/main/413_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Project Options
OPTION 1

For this project, each student is expected to explore a plausible and relevant public health research
question using one of the data sets outlined below. To achieve this, you will need to conduct a basic
literature review (3-5 references) to identify a potential research hypothesis and related variables; select
and acquire an appropriate data set; perform an exploratory data analysis (Table 1, graphs, etc.); and
run an appropriate multiple regression model of your hypothesis adjusting for possible confounders
(and possibly effect modifiers)

In [23]:
library(tidyverse)
library(ggplot2)

1. Download and review the descriptions of the National Health and Nutrition Examination Survey
(NHANES): This study is designed to assess the health and nutritional status of adults and
children in the United States. The survey is unique in that it combines interviews and physical
examinations, which means that it contains many measurement (numeric) variables. This is a
complex data set in that it’s divided into multiple files that need to be merged; a guide in the
form of R code can be found at the bottom of the link below:
https://www.cdc.gov/nchs/tutorials/NHANES/Downloads/intro.htm#6

Note: You may use any year’s data or combine multiple years of data if appropriate. However,
it is recommended that you use the most recent year that contains the variables of interest
since not all years have the same set of survey/examination variables. Combining multiple
years may make it more difficult to weigh data appropriately but it is doable.

2. For your choice of data, look over the code book and overview of the data to identify potential
variables for your analysis. Use the variables to guide your literature review to identify a
research question of public health relevance. The review should outline the relevance and
motivation of the research question, provide the necessary background information – that is,
previous results, data set used, co-variables used, and references – and a brief statement about
what you expect to find. For example, if you wish to study the relationship between dietary
intake and heart disease, then your hypothesis should include the exact variables used within
the relevant data set (e.g., NHANES), how the variable is coded, and a statement along the lines
of “we expect that individuals who report consuming more chocolate have lower hypertension
(or decreased blood pressure).”

Research question: How does access to WIC benefits during pregnancy associated with large for gestational age?

Dataset: NHANES 2017 - March 2020

Exposure: Mother's participation in WIC during pregnancy

Outcome: Large for gestational age (larger than 9 lb) infant

Covariates: race/ethnicity, health insurance status, household income, age at birth, BMI, month during pregnancy enrolled in WIC, gestational diabetes, education level, smoking status

Expected Result: We expect to find that being born large for gestational age is less likely if the mother was enrolled in WIC during pregnancy.

1.	A table containing the univariate distribution of all variables considered including the outcome, primary exposure, and all covariates (confounders and effect modifiers). The covariates should be limited to no more than ten variables. The literature search should guide the choice of variables. However, variables such as age, gender (if appropriate), co-morbidities, competing exposures, etc., should be incorporated whenever appropriate.

In [24]:
### Import datasets

library(haven)

# race/ethncity, education level
demodf = read_xpt("https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DEMO.xpt")
# household income
incdf = read_xpt("https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_INQ.xpt")
# BMI
bmidf = read_xpt("https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BMX.xpt")
# WIC participation, month enrolled
wicdf = read_xpt("https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_FSQ.xpt")
# health insurance status
hiqdf = read_xpt("https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_HIQ.xpt")
# gestational diabetes, baby more than 9 lbs, age at birth of first and last child
repdf = read_xpt("https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_RHQ.xpt")
# smoking status
smqdf = read_xpt("https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_SMQ.xpt")

### Merge datasets

library(dplyr)

nhanes = full_join(demodf, incdf, by = "SEQN") %>%
  left_join(bmidf, by="SEQN") %>%
  left_join(wicdf, by="SEQN") %>%
  left_join(hiqdf, by="SEQN") %>%
  left_join(repdf, by="SEQN") %>%
  left_join(smqdf, by="SEQN")

### Only include variables that are needed for the analysis
nhanes = nhanes %>%
  select(c(SEQN, RHQ172, FSD652ZW, RIDRETH3, HIQ011, HIQ032A, INDFMMPI,
            RHD190, BMXBMI, FSD672ZW, RHQ162, DMDEDUC2, SMQ040))

### Handling missing data

# Replace 7s and 9s with NA
nhanes = nhanes %>%
  mutate(RHQ172 = ifelse(RHQ172 %in% c(7,9), NA, RHQ172)) %>%
  mutate(FSD652ZW = ifelse(FSD652ZW %in% c(7,9), NA, FSD652ZW)) %>%
  mutate(HIQ011 = ifelse(HIQ011 %in% c(7,9), NA, HIQ011)) %>%
  mutate(HIQ032A = ifelse(HIQ032A %in% c(77,99), NA, HIQ032A)) %>%
  mutate(RHD190 = ifelse(RHD190 %in% c(777,999), NA, RHD190)) %>%
  mutate(FSD672ZW = ifelse(FSD672ZW %in% c(77,99), NA, FSD672ZW)) %>%
  mutate(RHQ162 = ifelse(RHQ162 %in% c(7,9), NA, RHQ162)) %>%
  mutate(DMDEDUC2 = ifelse(DMDEDUC2 %in% c(7,9), NA, DMDEDUC2)) %>%
  mutate(SMQ040 = ifelse(SMQ040 %in% c(7,9), NA, SMQ040))

# Replace missing data with average value for continuous variables
nhanes = nhanes %>%
  mutate(INDFMMPI = ifelse(is.na(INDFMMPI), mean(nhanes$INDFMMPI, na.rm=T),
                           INDFMMPI)) %>%
  mutate(BMXBMI = ifelse(is.na(BMXBMI), mean(nhanes$BMXBMI, na.rm=T),
                         BMXBMI)) %>%
  mutate(RHD190 = ifelse(is.na(RHD190), mean(nhanes$RHD190, na.rm=T),
                         RHD190)) %>%
  mutate(FSD672ZW = ifelse(is.na(FSD672ZW), mean(nhanes$FSD672ZW, na.rm=T),
                           FSD672ZW))

### Exclusion criteria (only income patients with outcome and main exposure data)
nhanes = nhanes %>%
  filter(!is.na(RHQ172)) %>%
  filter(!is.na(FSD652ZW))

### Sample size after exclusion criteria and missing data removed
nrow(nhanes)

### Redefine variables

# Create new variable HINSTYPE: 1 = Private insurance, 2 = Government insurance,
# 3 = No insurance
nhanes = nhanes %>%
  mutate(HINSTYPE = NA) %>%
  mutate(HINSTYPE = ifelse(HIQ032A==1, 1, HINSTYPE)) %>%
  mutate(HINSTYPE = ifelse(HIQ011==2, 3, HINSTYPE)) %>%
  mutate(HINSTYPE = ifelse(HIQ011==1 & is.na(HIQ032A), 2, HINSTYPE))

# Convert categorical variables into factor variables with labels
nhanes = nhanes %>%
  mutate(RHQ172 = factor(RHQ172, labels = c("Yes", "No"))) %>%
  mutate(FSD652ZW = factor(FSD652ZW, labels = c("Yes", "No"))) %>%
  mutate(HINSTYPE = factor(HINSTYPE, labels = c("Private insurance",
                                                "Government insurance",
                                                "No insurance"))) %>%
  mutate(RIDRETH3 = factor(RIDRETH3, labels = c("Hispanic", "Hispanic",
                                                "White", "Black", "Asian",
                                                "Multiracial"))) %>%
  mutate(DMDEDUC2 = cut(DMDEDUC2, breaks = c(1, 2, 3, 4, 5, 6),
                        labels = c("<9th grade", "9-11th grade", "High school",
                                   "Some college", ">=College graduate"),
                        right = F, include.lowest = T)) %>%
  mutate(RHQ162 = factor(RHQ162, labels = c("Yes", "No"))) %>%
  mutate(SMQ040 = factor(SMQ040, labels = c("Every day", "Some days",
                                            "Not at all")))
nhanes

# List proportion of those who had LGA baby vs. those who didn't
proportions(table(nhanes$RHQ172))

# List proportion of those who participated WIC vs. those who didn't
proportions(table(nhanes$FSD652ZW))



[1] 279

SEQN,RHQ172,FSD652ZW,RIDRETH3,HIQ011,HIQ032A,INDFMMPI,RHD190,BMXBMI,FSD672ZW,RHQ162,DMDEDUC2,SMQ040,HINSTYPE
<dbl>,<fct>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<fct>,<fct>
109396,No,No,Asian,1,1,1.840000,35.00000,21.5,3.744526,No,>=College graduate,NA,Private insurance
109432,No,No,Black,1,1,3.140000,43.00000,19.8,3.744526,No,>=College graduate,NA,Private insurance
109447,No,No,White,2,NA,0.520000,23.00000,26.3,3.744526,No,Some college,Every day,No insurance
109478,No,No,White,1,1,4.110000,27.00000,26.1,3.744526,No,Some college,NA,Private insurance
109541,No,No,Black,1,1,1.960000,34.00000,32.7,3.744526,No,Some college,NA,Private insurance
109687,Yes,No,White,1,1,2.570000,36.00000,21.3,3.744526,No,Some college,NA,Private insurance
109693,No,Yes,Hispanic,2,NA,0.730000,25.00000,30.5,1.000000,Yes,Some college,NA,No insurance
109716,No,No,Black,1,NA,0.590000,24.00000,33.1,3.744526,No,9-11th grade,Every day,Government insurance
109789,No,No,White,1,1,5.000000,30.00000,44.7,3.744526,No,>=College graduate,NA,Private insurance



      Yes        No 
0.1003584 0.8996416 


      Yes        No 
0.4910394 0.5089606 

In [25]:
nhanes1 = nhanes

In [26]:
# Drop the original insurance columns
nhanes1 <- subset(nhanes1, select = -c(HIQ011, HIQ032A))
nhanes1

SEQN,RHQ172,FSD652ZW,RIDRETH3,INDFMMPI,RHD190,BMXBMI,FSD672ZW,RHQ162,DMDEDUC2,SMQ040,HINSTYPE
<dbl>,<fct>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<fct>,<fct>
109396,No,No,Asian,1.840000,35.00000,21.5,3.744526,No,>=College graduate,NA,Private insurance
109432,No,No,Black,3.140000,43.00000,19.8,3.744526,No,>=College graduate,NA,Private insurance
109447,No,No,White,0.520000,23.00000,26.3,3.744526,No,Some college,Every day,No insurance
109478,No,No,White,4.110000,27.00000,26.1,3.744526,No,Some college,NA,Private insurance
109541,No,No,Black,1.960000,34.00000,32.7,3.744526,No,Some college,NA,Private insurance
109687,Yes,No,White,2.570000,36.00000,21.3,3.744526,No,Some college,NA,Private insurance
109693,No,Yes,Hispanic,0.730000,25.00000,30.5,1.000000,Yes,Some college,NA,No insurance
109716,No,No,Black,0.590000,24.00000,33.1,3.744526,No,9-11th grade,Every day,Government insurance
109789,No,No,White,5.000000,30.00000,44.7,3.744526,No,>=College graduate,NA,Private insurance


In [27]:
library(dplyr)

# Define a mapping from new name to old name
rename_map <- c(
  ID = "SEQN",
  LGA = "RHQ172",
  WIC_ben = "FSD652ZW",
  RACE_ETH = "RIDRETH3",
  HH_INCOME = "INDFMMPI",
  MAGE_LB = "RHD190",
  M_BMI = "BMXBMI",
  WIC_ENROLL_MP = "FSD672ZW",
  GEST_DB = "RHQ162",
  M_EDUC = "DMDEDUC2",
  M_SMOKE = "SMQ040"
)

# Get the current column names in nhanes1
current_cols <- names(nhanes1)

# Identify which of the 'old_name' columns still exist in the dataframe
# and construct the renaming arguments as 'new_name = old_name'
renames_to_apply <- c()
for (new_name in names(rename_map)) {
  old_name <- rename_map[new_name]
  if (old_name %in% current_cols) {
    renames_to_apply[new_name] <- old_name
  }
}

# Apply renaming only for columns that exist
if (length(renames_to_apply) > 0) {
  nhanes1 <- nhanes1 %>% rename(!!!renames_to_apply)
} else {
  message("All relevant columns are already renamed or do not exist in the dataframe.")
}

nhanes1

ID,LGA,WIC_ben,RACE_ETH,HH_INCOME,MAGE_LB,M_BMI,WIC_ENROLL_MP,GEST_DB,M_EDUC,M_SMOKE,HINSTYPE
<dbl>,<fct>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<fct>,<fct>
109396,No,No,Asian,1.840000,35.00000,21.5,3.744526,No,>=College graduate,NA,Private insurance
109432,No,No,Black,3.140000,43.00000,19.8,3.744526,No,>=College graduate,NA,Private insurance
109447,No,No,White,0.520000,23.00000,26.3,3.744526,No,Some college,Every day,No insurance
109478,No,No,White,4.110000,27.00000,26.1,3.744526,No,Some college,NA,Private insurance
109541,No,No,Black,1.960000,34.00000,32.7,3.744526,No,Some college,NA,Private insurance
109687,Yes,No,White,2.570000,36.00000,21.3,3.744526,No,Some college,NA,Private insurance
109693,No,Yes,Hispanic,0.730000,25.00000,30.5,1.000000,Yes,Some college,NA,No insurance
109716,No,No,Black,0.590000,24.00000,33.1,3.744526,No,9-11th grade,Every day,Government insurance
109789,No,No,White,5.000000,30.00000,44.7,3.744526,No,>=College graduate,NA,Private insurance


In [28]:
install.packages("table1")
library(table1)

table1(~ LGA +
  WIC_ben +
  RACE_ETH +
  HH_INCOME +
  MAGE_LB +
  M_BMI +
  WIC_ENROLL_MP +
  GEST_DB +
  M_EDUC +
  M_SMOKE | WIC_ben, data=nhanes1)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘Formula’



Attaching package: ‘table1’


The following objects are masked from ‘package:base’:

    units, units<-




<table class="Rtable1">
<thead>
<tr>
<th class='rowlabel firstrow lastrow'></th>
<th class='firstrow lastrow'><span class='stratlabel'>Yes<br/><span class='stratn'>(N=137)</span></span></th>
<th class='firstrow lastrow'><span class='stratlabel'>No<br/><span class='stratn'>(N=142)</span></span></th>
<th class='firstrow lastrow'><span class='stratlabel'>Overall<br/><span class='stratn'>(N=279)</span></span></th>
</tr>
</thead>
<tbody>
<tr>
<td class='rowlabel firstrow'><span class='varlabel'>LGA</span></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
</tr>
<tr>
<td class='rowlabel'>Yes</td>
<td>14 (10.2%)</td>
<td>14 (9.9%)</td>
<td>28 (10.0%)</td>
</tr>
<tr>
<td class='rowlabel lastrow'>No</td>
<td class='lastrow'>123 (89.8%)</td>
<td class='lastrow'>128 (90.1%)</td>
<td class='lastrow'>251 (90.0%)</td>
</tr>
<tr>
<td class='rowlabel firstrow'><span class='varlabel'>WIC_ben</span></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
</tr>
<tr>
<td class='rowlabel'>Yes</td>
<td>137 (100%)</td>
<td>0 (0%)</td>
<td>137 (49.1%)</td>
</tr>
<tr>
<td class='rowlabel lastrow'>No</td>
<td class='lastrow'>0 (0%)</td>
<td class='lastrow'>142 (100%)</td>
<td class='lastrow'>142 (50.9%)</td>
</tr>
<tr>
<td class='rowlabel firstrow'><span class='varlabel'>RACE_ETH</span></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
</tr>
<tr>
<td class='rowlabel'>Hispanic</td>
<td>49 (35.8%)</td>
<td>26 (18.3%)</td>
<td>75 (26.9%)</td>
</tr>
<tr>
<td class='rowlabel'>White</td>
<td>28 (20.4%)</td>
<td>55 (38.7%)</td>
<td>83 (29.7%)</td>
</tr>
<tr>
<td class='rowlabel'>Black</td>
<td>43 (31.4%)</td>
<td>41 (28.9%)</td>
<td>84 (30.1%)</td>
</tr>
<tr>
<td class='rowlabel'>Asian</td>
<td>7 (5.1%)</td>
<td>11 (7.7%)</td>
<td>18 (6.5%)</td>
</tr>
<tr>
<td class='rowlabel lastrow'>Multiracial</td>
<td class='lastrow'>10 (7.3%)</td>
<td class='lastrow'>9 (6.3%)</td>
<td class='lastrow'>19 (6.8%)</td>
</tr>
<tr>
<td class='rowlabel firstrow'><span class='varlabel'>HH_INCOME</span></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
</tr>
<tr>
<td class='rowlabel'>Mean (SD)</td>
<td>1.31 (0.773)</td>
<td>2.16 (1.45)</td>
<td>1.75 (1.24)</td>
</tr>
<tr>
<td class='rowlabel lastrow'>Median [Min, Max]</td>
<td class='lastrow'>1.20 [0, 3.18]</td>
<td class='lastrow'>2.18 [0, 5.00]</td>
<td class='lastrow'>1.47 [0, 5.00]</td>
</tr>
<tr>
<td class='rowlabel firstrow'><span class='varlabel'>MAGE_LB</span></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
</tr>
<tr>
<td class='rowlabel'>Mean (SD)</td>
<td>27.2 (5.83)</td>
<td>28.9 (5.85)</td>
<td>28.1 (5.90)</td>
</tr>
<tr>
<td class='rowlabel lastrow'>Median [Min, Max]</td>
<td class='lastrow'>26.0 [17.0, 42.0]</td>
<td class='lastrow'>29.0 [17.0, 43.0]</td>
<td class='lastrow'>27.0 [17.0, 43.0]</td>
</tr>
<tr>
<td class='rowlabel firstrow'><span class='varlabel'>M_BMI</span></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
</tr>
<tr>
<td class='rowlabel'>Mean (SD)</td>
<td>30.5 (7.43)</td>
<td>30.3 (7.75)</td>
<td>30.4 (7.58)</td>
</tr>
<tr>
<td class='rowlabel lastrow'>Median [Min, Max]</td>
<td class='lastrow'>29.4 [16.2, 52.3]</td>
<td class='lastrow'>29.5 [18.4, 53.1]</td>
<td class='lastrow'>29.4 [16.2, 53.1]</td>
</tr>
<tr>
<td class='rowlabel firstrow'><span class='varlabel'>WIC_ENROLL_MP</span></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
</tr>
<tr>
<td class='rowlabel'>Mean (SD)</td>
<td>3.73 (2.03)</td>
<td>3.74 (0)</td>
<td>3.74 (1.42)</td>
</tr>
<tr>
<td class='rowlabel lastrow'>Median [Min, Max]</td>
<td class='lastrow'>3.00 [1.00, 9.00]</td>
<td class='lastrow'>3.74 [3.74, 3.74]</td>
<td class='lastrow'>3.74 [1.00, 9.00]</td>
</tr>
<tr>
<td class='rowlabel firstrow'><span class='varlabel'>GEST_DB</span></td>
<td class='firstrow'></td>
<td class='firstrow'>

In [34]:
install.packages("htmltools")
library(htmltools)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [39]:
library(htmltools)
table_html = table1(~ LGA + WIC_ben + RACE_ETH + HH_INCOME + MAGE_LB + M_BMI + WIC_ENROLL_MP + GEST_DB + M_EDUC + M_SMOKE | WIC_ben, data=nhanes1)

# Display the HTML table directly
table_html

<table class="Rtable1">
<thead>
<tr>
<th class='rowlabel firstrow lastrow'></th>
<th class='firstrow lastrow'><span class='stratlabel'>Yes<br/><span class='stratn'>(N=137)</span></span></th>
<th class='firstrow lastrow'><span class='stratlabel'>No<br/><span class='stratn'>(N=142)</span></span></th>
<th class='firstrow lastrow'><span class='stratlabel'>Overall<br/><span class='stratn'>(N=279)</span></span></th>
</tr>
</thead>
<tbody>
<tr>
<td class='rowlabel firstrow'><span class='varlabel'>LGA</span></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
</tr>
<tr>
<td class='rowlabel'>Yes</td>
<td>14 (10.2%)</td>
<td>14 (9.9%)</td>
<td>28 (10.0%)</td>
</tr>
<tr>
<td class='rowlabel lastrow'>No</td>
<td class='lastrow'>123 (89.8%)</td>
<td class='lastrow'>128 (90.1%)</td>
<td class='lastrow'>251 (90.0%)</td>
</tr>
<tr>
<td class='rowlabel firstrow'><span class='varlabel'>WIC_ben</span></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
</tr>
<tr>
<td class='rowlabel'>Yes</td>
<td>137 (100%)</td>
<td>0 (0%)</td>
<td>137 (49.1%)</td>
</tr>
<tr>
<td class='rowlabel lastrow'>No</td>
<td class='lastrow'>0 (0%)</td>
<td class='lastrow'>142 (100%)</td>
<td class='lastrow'>142 (50.9%)</td>
</tr>
<tr>
<td class='rowlabel firstrow'><span class='varlabel'>RACE_ETH</span></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
</tr>
<tr>
<td class='rowlabel'>Hispanic</td>
<td>49 (35.8%)</td>
<td>26 (18.3%)</td>
<td>75 (26.9%)</td>
</tr>
<tr>
<td class='rowlabel'>White</td>
<td>28 (20.4%)</td>
<td>55 (38.7%)</td>
<td>83 (29.7%)</td>
</tr>
<tr>
<td class='rowlabel'>Black</td>
<td>43 (31.4%)</td>
<td>41 (28.9%)</td>
<td>84 (30.1%)</td>
</tr>
<tr>
<td class='rowlabel'>Asian</td>
<td>7 (5.1%)</td>
<td>11 (7.7%)</td>
<td>18 (6.5%)</td>
</tr>
<tr>
<td class='rowlabel lastrow'>Multiracial</td>
<td class='lastrow'>10 (7.3%)</td>
<td class='lastrow'>9 (6.3%)</td>
<td class='lastrow'>19 (6.8%)</td>
</tr>
<tr>
<td class='rowlabel firstrow'><span class='varlabel'>HH_INCOME</span></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
</tr>
<tr>
<td class='rowlabel'>Mean (SD)</td>
<td>1.31 (0.773)</td>
<td>2.16 (1.45)</td>
<td>1.75 (1.24)</td>
</tr>
<tr>
<td class='rowlabel lastrow'>Median [Min, Max]</td>
<td class='lastrow'>1.20 [0, 3.18]</td>
<td class='lastrow'>2.18 [0, 5.00]</td>
<td class='lastrow'>1.47 [0, 5.00]</td>
</tr>
<tr>
<td class='rowlabel firstrow'><span class='varlabel'>MAGE_LB</span></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
</tr>
<tr>
<td class='rowlabel'>Mean (SD)</td>
<td>27.2 (5.83)</td>
<td>28.9 (5.85)</td>
<td>28.1 (5.90)</td>
</tr>
<tr>
<td class='rowlabel lastrow'>Median [Min, Max]</td>
<td class='lastrow'>26.0 [17.0, 42.0]</td>
<td class='lastrow'>29.0 [17.0, 43.0]</td>
<td class='lastrow'>27.0 [17.0, 43.0]</td>
</tr>
<tr>
<td class='rowlabel firstrow'><span class='varlabel'>M_BMI</span></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
</tr>
<tr>
<td class='rowlabel'>Mean (SD)</td>
<td>30.5 (7.43)</td>
<td>30.3 (7.75)</td>
<td>30.4 (7.58)</td>
</tr>
<tr>
<td class='rowlabel lastrow'>Median [Min, Max]</td>
<td class='lastrow'>29.4 [16.2, 52.3]</td>
<td class='lastrow'>29.5 [18.4, 53.1]</td>
<td class='lastrow'>29.4 [16.2, 53.1]</td>
</tr>
<tr>
<td class='rowlabel firstrow'><span class='varlabel'>WIC_ENROLL_MP</span></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
<td class='firstrow'></td>
</tr>
<tr>
<td class='rowlabel'>Mean (SD)</td>
<td>3.73 (2.03)</td>
<td>3.74 (0)</td>
<td>3.74 (1.42)</td>
</tr>
<tr>
<td class='rowlabel lastrow'>Median [Min, Max]</td>
<td class='lastrow'>3.00 [1.00, 9.00]</td>
<td class='lastrow'>3.74 [3.74, 3.74]</td>
<td class='lastrow'>3.74 [1.00, 9.00]</td>
</tr>
<tr>
<td class='rowlabel firstrow'><span class='varlabel'>GEST_DB</span></td>
<td class='firstrow'></td>
<td class='firstrow'>

References:
*   Hamad, R., Collin, D. F., Baer, R. J., & Jelliffe-Pawlowski, L. L. (2019). Association of Revised WIC Food Package With Perinatal and Birth Outcomes: A Quasi-Experimental Study. JAMA pediatrics, 173(9), 845–852. https://doi.org/10.1001/jamapediatrics.2019.1706
*   Ratnasiri, A. W. G., Parry, S. S., Arief, V. N., DeLacy, I. H., Lakshminrusimha, S., Halliday, L. A., DiLibero, R. J., & Basford, K. E. (2018). Temporal trends, patterns, and predictors of preterm birth in California from 2007 to 2016, based on the obstetric estimate of gestational age. Maternal health, neonatology and perinatology, 4, 25. https://doi.org/10.1186/s40748-018-0094-0
*   Venkatesh, K. K., Huang, X., Cameron, N. A., Petito, L. C., Garner, J., Headings, A., Hanks, A. S., Grobman, W. A., & Khan, S. S. (2024). Special Supplemental Nutrition Program for Women, Infants, and Children Enrollment and Adverse Pregnancy Outcomes Among Nulliparous Individuals. Obstetrics and gynecology, 144(2), 223–232. https://doi.org/10.1097/AOG.0000000000005660



3. In some cases (e.g., when using NHANES), the sample sizes in some cells may be very small due
to the outcome being very rare in the population. To mitigate this issue, you may need to
combine multiple cycles (years) of the data sets and modify the sample weights appropriately.

4. The results should be in the form presentation slides (PowerPoint or PDF). While you will not be
presenting your results, the slides need to correspond to a 15-minute presentation given at a
public health conference. Please also include a descriptive title, references using accepted
bibliography style (e.g., Chicago). All charts, figures, and tables should be clearly labeled with
titles, legends, or footnotes.


2.	Provide accompanying plots and figures to summarize important findings. Figures (as well as tables) should be descriptive and aid in the intended narrative of the project. Use legends, colors, and titles. There are packages that allow you to plot results from a linear model (http://strengejacke.de/sjPlot/sjp.glm/).

3.	Tabulate the distribution of each variable among those who have the outcome vs. those who do not. If your outcome is continuous then create a temporary dichotomous variable to tabulate the bivariate relationships. If both outcome and exposure are continuous, then provide the correlation value. Use p-values to establish the significance of each bivariate association, include the association measure (e.g., odds ratio) and its confidence interval.

4.	Create an appropriate regression model to examine the relationship between the outcome and exposure, controlling for important covariates. As an optional step, check for effect modification using interaction terms. If you do this, however, you will need to display your results appropriately. You are encouraged to conduct some form of model selection to include consequential variables.

5.	OPTIONAL: The surveys used for this analysis are meant to be representative of the entire population. However, the sampling approach will lead to either over- or under-representation of certain groups. To account for this, the sample should be appropriately weighted using the weight variables provided in the data set. The package survey provides a set of methods for achieving this.

6.	Provide a concise summary of the analysis and results (include any interesting observations as well). Based on your analysis, what can you conclude regarding the research question? What are some strengths and limitations of your analysis? Limitations typically include missing data; definition of variables (does the outcome variable fairly represent the research endpoint? For example: is blood HbA1C a sufficiently good metric for diabetes?); were there any violations of parametric assumptions?